# Allentown, Pennsylvania — Full LVT Model

Models a shift from Allentown's existing split-rate property tax to full Land Value Tax (buildings exempt, all revenue on land).

**Data source**: Lehigh County ArcGIS FeatureServer — Layer 0 (assessment values) + Layer 1 (parcel polygons), joined on PIN.  
**Current system**: City split-rate — land taxed at 24.4705 mills, improvements at 4.6293 mills (≈5.3:1 ratio), per adopted 2026 budget (~3.96% increase over 2024).  
**Reform**: Full LVT — improvement millage → 0, land millage recalculated to maintain revenue neutrality.  
**Revenue baseline**: Derived from 2026 millages × 2024/25 assessment roll (2026 ACFR not yet published). 2024 ACFR levy was $39,689,350.

In [1]:
import sys
import json
import os
import time
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from shapely.geometry import Polygon, MultiPolygon
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

# Constants
CITY_NAME = 'allentown'
STATE_FIPS = '42'       # Pennsylvania
COUNTY_FIPS = '077'     # Lehigh County
MODEL_TYPE = 'full_lvt'

# 2026 adopted city millage rates (budget approved with ~3.96% increase over 2024)
# Source: City of Allentown 2026 Final Budget Appendix
LAND_MILLAGE_CURRENT = 24.4705
IMP_MILLAGE_CURRENT  =  4.6293

# OFFICIAL_REVENUE derived below from 2026 millages × assessment roll
# (2026 ACFR not yet published; 2024 ACFR levy was $39,689,350)

BASE_URL = 'https://services1.arcgis.com/XWDNR4PQlDQwrRCL/arcgis/rest/services/ATestParcel/FeatureServer'

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print('Setup complete.')

Setup complete.


## Section 2 — Fetch / Load Parcel Data

Two layers joined on PIN:
- **Layer 1**: Parcel polygon geometry, filtered to `MUNI_CODE='02'` (City of Allentown)
- **Layer 0**: Assessment values (`TAXLND`, `TAXBLD`, `TOTAPR`, `TAXAPR`, `CLASS`, `USELAND`), filtered to `DIST='02'`

In [2]:
def fetch_arcgis_features(base_url, layer_id, where_clause, out_fields='*', with_geometry=True, page_size=1000):
    """Paginate an ArcGIS FeatureServer layer with a WHERE filter."""
    url = f'{base_url}/{layer_id}/query'

    r = requests.get(url, params={'where': where_clause, 'returnCountOnly': 'true', 'f': 'json'}, timeout=30)
    total = r.json()['count']
    print(f'  Layer {layer_id}: {total:,} records')

    rows = []
    geoms = []
    for offset in range(0, total, page_size):
        resp = requests.get(url, params={
            'where': where_clause,
            'outFields': out_fields,
            'returnGeometry': str(with_geometry).lower(),
            'resultOffset': offset,
            'resultRecordCount': page_size,
            'outSR': '4326',
            'f': 'json',
        }, timeout=60)
        features = resp.json().get('features', [])
        for feat in features:
            attrs = feat['attributes']
            if with_geometry:
                geom_json = feat.get('geometry')
                if geom_json and 'rings' in geom_json:
                    rings = geom_json['rings']
                    polys = [Polygon(r) for r in rings if len(r) >= 3]
                    attrs['_geom'] = polys[0] if len(polys) == 1 else (MultiPolygon(polys) if polys else None)
                else:
                    attrs['_geom'] = None
            rows.append(attrs)
        if (offset // page_size) % 10 == 0:
            print(f'    {min(offset + page_size, total):,}/{total:,}...')

    return rows

In [3]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'

if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f'Loaded {len(gdf):,} parcels from cache')
else:
    # Layer 1 — polygon geometry
    print('Downloading Layer 1 (geometry)...')
    rows1 = fetch_arcgis_features(
        BASE_URL, layer_id=1,
        where_clause="MUNI_CODE='02'",
        out_fields='PIN,MUNI_CODE,ACRES,SITUS_ADDR_NUM,SITUS_FNAME,SITUS_FTYPE,SITUS_CITY,SITUS_ZIP',
        with_geometry=True,
    )
    df1 = pd.DataFrame(rows1)
    gdf1 = gpd.GeoDataFrame(df1, geometry='_geom', crs='EPSG:4326')
    gdf1 = gdf1.rename(columns={'_geom': 'geometry'}).set_geometry('geometry')
    gdf1 = gdf1[gdf1.geometry.notna()].copy()
    print(f'  Layer 1: {len(gdf1):,} parcels with geometry')

    # Layer 0 — assessment attributes
    print('Downloading Layer 0 (assessment values)...')
    rows0 = fetch_arcgis_features(
        BASE_URL, layer_id=0,
        where_clause="DIST='02'",
        out_fields='PIN,DIST,CLASS,USELAND,TAXLND,TAXBLD,TOTAPR,TAXAPR,TOTASMT,TAXASMT,BLDEXPT,LNDEXPT,NAMOWN,ADDRESS,UNITS',
        with_geometry=False,
    )
    df0 = pd.DataFrame(rows0)
    print(f'  Layer 0: {len(df0):,} assessment records')

    # Join geometry (Layer 1) to assessment values (Layer 0) on PIN
    gdf = gdf1.merge(df0, on='PIN', how='left')
    print(f'After join: {len(gdf):,} parcels, {gdf["TAXLND"].isna().sum():,} missing Layer 0 data')

    gdf.to_parquet(PARCEL_PATH)
    print(f'Cached to {PARCEL_PATH}')

print(f'\nColumns: {list(gdf.columns)}')
print(gdf[['TAXLND', 'TAXBLD', 'TOTAPR', 'TAXAPR']].describe())

Loaded 34,393 parcels from cache

Columns: ['PIN', 'MUNI_CODE', 'ACRES', 'SITUS_ADDR_NUM', 'SITUS_FNAME', 'SITUS_FTYPE', 'SITUS_CITY', 'SITUS_ZIP', 'geometry', 'DIST', 'CLASS', 'USELAND', 'TAXLND', 'TAXBLD', 'TOTAPR', 'TAXAPR', 'TOTASMT', 'TAXASMT', 'BLDEXPT', 'LNDEXPT', 'NAMOWN', 'ADDRESS', 'UNITS']
             TAXLND        TAXBLD        TOTAPR        TAXAPR
count  3.439100e+04  3.439100e+04  3.439100e+04  3.439100e+04
mean   2.328154e+04  1.388060e+05  2.110192e+05  1.620876e+05
std    8.742090e+04  6.620531e+05  1.389939e+06  7.091921e+05
min    0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
25%    6.000000e+03  6.390000e+04  7.420000e+04  7.110000e+04
50%    9.800000e+03  8.620000e+04  1.023000e+05  1.001000e+05
75%    2.380000e+04  1.169000e+05  1.413000e+05  1.380000e+05
max    4.707200e+06  4.550960e+07  1.179743e+08  4.585770e+07


## Section 3 — Classify and Validate

**Column mapping:**

| Concept | Column | Notes |
|---|---|---|
| Taxable land value | `TAXLND` | Post-exemption assessed land value |
| Taxable improvement value | `TAXBLD` | Post-exemption assessed building value |
| Total appraised | `TOTAPR` | Pre-exemption total market value |
| Total taxable | `TAXAPR` | Post-exemption taxable total (`TAXLND + TAXBLD`) |
| Property class | `CLASS` | R=Residential, A=Apartment, C=Commercial, I=Industrial, V=Vacant, S=Exempt |
| Use code | `USELAND` | STEB sub-classification |
| Exemption (land) | `LNDEXPT` | Dollar value of land exemption |
| Exemption (bldg) | `BLDEXPT` | Dollar value of building exemption |

**Assessment basis**: 100% of market value (Pennsylvania full-value assessment). 2024 ACFR confirms assessed = estimated actual value (Table 6).

**Millage rates** (2026 adopted budget): Land = 24.4705 mills, Improvements = 4.6293 mills (~3.96% increase over 2024 rates of 23.5376 / 4.4528).  
Combined equivalent: ~7.5679 mills.

In [4]:
# Fill missing assessment values (parcels in Layer 1 not found in Layer 0)
for col in ['TAXLND', 'TAXBLD', 'TOTAPR', 'TAXAPR', 'BLDEXPT', 'LNDEXPT']:
    gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0)

# Taxable value columns (already post-exemption from the source)
gdf['taxable_land_value'] = gdf['TAXLND'].clip(lower=0)
gdf['taxable_improvement_value'] = gdf['TAXBLD'].clip(lower=0)

# Full exemption flag: taxable total is zero but appraised value is positive
gdf['full_exmp'] = ((gdf['TAXAPR'] == 0) & (gdf['TOTAPR'] > 0)).astype(int)

# Property category mapping using CLASS letter codes
# PA STEB USELAND sub-codes for residential:
#   1101 = single-family detached, 1102 = semi-detached, 1103 = row/townhouse,
#   1193 = residential condo, 1198/1199 = other residential

def classify_parcel(row):
    cls = str(row.get('CLASS', '')).strip()
    useland = str(row.get('USELAND', '')).strip()
    taxbld = row.get('TAXBLD', 0) or 0

    if cls == 'S':
        return 'Exempt'
    if cls == 'V':
        return 'Vacant Land'
    if cls == 'R':
        if useland == '1103':
            return 'Townhome / Rowhouse'
        if useland == '1193':
            return 'Condominium'
        return 'Single Family Residential'
    if cls == 'A':
        return 'Large Multi-Family (5+ units)'
    if cls == 'C':
        return 'Commercial'
    if cls == 'I':
        return 'Industrial'
    # Catch-all: use appraised values to distinguish
    if taxbld <= 0:
        return 'Vacant Land'
    return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf.apply(classify_parcel, axis=1)

# Override: zero improvement → Vacant Land (catches coded-as-R/C/I but unimproved)
gdf.loc[gdf['taxable_improvement_value'] <= 0, 'PROPERTY_CATEGORY'] = gdf.loc[
    gdf['taxable_improvement_value'] <= 0, 'PROPERTY_CATEGORY'
].replace({'Single Family Residential': 'Vacant Land',
           'Townhome / Rowhouse': 'Vacant Land',
           'Commercial': 'Vacant Land',
           'Industrial': 'Vacant Land',
           'Other': 'Vacant Land'})
# Exempt parcels keep their exempt status regardless of improvement value
gdf.loc[(gdf['full_exmp'] == 1) & (gdf['PROPERTY_CATEGORY'] != 'Exempt'), 'PROPERTY_CATEGORY'] = 'Exempt'

print('PROPERTY_CATEGORY distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts())
print(f'\nFully exempt: {gdf["full_exmp"].sum():,}')
print(f'Parcels with $0 TAXLND: {(gdf["taxable_land_value"] == 0).sum():,}')

PROPERTY_CATEGORY distribution:
PROPERTY_CATEGORY
Single Family Residential        21159
Townhome / Rowhouse               8394
Commercial                        1842
Exempt                            1392
Vacant Land                        690
Industrial                         440
Large Multi-Family (5+ units)      401
Condominium                         72
Other                                3
Name: count, dtype: int64

Fully exempt: 1,312
Parcels with $0 TAXLND: 1,585


## Section 4 — Current Tax Model

Current tax = `TAXLND × 24.4705/1000 + TAXBLD × 4.6293/1000` (2026 adopted rates).

`OFFICIAL_REVENUE` is derived from these millages × the assessment roll (2026 ACFR not yet published).
Exempt parcels (TAXAPR=0, TOTAPR>0) pay no tax and are excluded from revenue computation.

In [5]:
# Current tax under 2026 split-rate
gdf['current_tax'] = (
    gdf['taxable_land_value'] * LAND_MILLAGE_CURRENT / 1000
    + gdf['taxable_improvement_value'] * IMP_MILLAGE_CURRENT / 1000
)
# Exempt parcels: zero tax
gdf.loc[gdf['full_exmp'] == 1, 'current_tax'] = 0.0

current_revenue = gdf['current_tax'].sum()
ref_2024_levy = 39_689_350
print(f'Modeled 2026 revenue (derived): ${current_revenue:>14,.0f}')
print(f'2024 ACFR levy (reference):     ${ref_2024_levy:>14,.0f}')
print(f'Year-over-year change:          {(current_revenue / ref_2024_levy - 1)*100:+.2f}%')

print(f'\nTaxable land base:         ${gdf.loc[gdf["full_exmp"]==0, "taxable_land_value"].sum():>14,.0f}')
print(f'Taxable improvement base:  ${gdf.loc[gdf["full_exmp"]==0, "taxable_improvement_value"].sum():>14,.0f}')

Modeled 2026 revenue (derived): $    41,691,716
2024 ACFR levy (reference):     $    39,689,350
Year-over-year change:          +5.05%

Taxable land base:         $   800,675,300
Taxable improvement base:  $ 4,773,678,800


## Section 5 — Full LVT Reform

All city revenue shifts to land. Improvement millage → 0. Land millage recalculated as:

```
new_land_millage = current_revenue / total_taxable_land × 1000
```

Revenue neutral by construction.

In [6]:
# Full LVT: all revenue onto land, buildings fully exempt
taxable = gdf[gdf['full_exmp'] == 0].copy()

total_taxable_land = taxable['taxable_land_value'].sum()
taxable_revenue = taxable['current_tax'].sum()

land_millage = (taxable_revenue / total_taxable_land) * 1000
improvement_millage = 0.0

taxable['new_tax'] = taxable['taxable_land_value'] * land_millage / 1000
taxable['tax_change'] = taxable['new_tax'] - taxable['current_tax']
taxable['tax_change_pct'] = (
    taxable['tax_change'] / taxable['current_tax'].replace(0, float('nan'))
) * 100

# Recombine with exempt parcels
exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0

gdf = pd.concat([taxable, exempt]).sort_index()

new_revenue = gdf['new_tax'].sum()
print(f'Land millage (Full LVT):    {land_millage:.4f} mills')
print(f'Current land millage:       {LAND_MILLAGE_CURRENT:.4f} mills')
print(f'Improvement millage:        {improvement_millage:.4f} mills (was {IMP_MILLAGE_CURRENT})')
print(f'Revenue check:              ${new_revenue:,.0f} (target: ${taxable_revenue:,.0f})')

# Category summary
category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Allentown — Full LVT Impact')

Land millage (Full LVT):    52.0707 mills
Current land millage:       24.4705 mills
Improvement millage:        0.0000 mills (was 4.6293)
Revenue check:              $41,691,716 (target: $41,691,716)

Allentown — Full LVT Impact
                     Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
    Single Family Residential  21159       $-158,356       -0.8%        $-7         $-59   -6.8%      -8.2%            31.7%            48.5%
          Townhome / Rowhouse   8394     $-1,343,609      -33.1%      $-160        $-153  -32.7%     -33.7%             0.3%            97.5%
                   Commercial   1842      $1,318,538       13.9%       $716         $246   19.9%      15.1%            53.7%            32.6%
                       Exempt   1392         $49,205       15.9%        $35           $0    0.0%       0.0%             2.9%             2.4%
                  Vacant Land    690        $680,505      112

## Section 6 — Exploration Charts

In [7]:
fig, ax = plt.subplots(figsize=(10, 6))
non_exempt = gdf[gdf['full_exmp'] == 0]
summary = non_exempt.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values()
colors = ['#d73027' if v > 0 else '#4575b4' for v in summary.values]
summary.plot.barh(ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Allentown — Median Tax Change % by Category (Full LVT)', fontsize=12)
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print('Saved category_preview.png')

Saved category_preview.png

## Section 7 — Census Join + Export

In [8]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


In [9]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print('Done.')

  ✓ allentown: 34,393 rows → ../../analysis/data/allentown.csv  [model: full_lvt]


Done.


## Validation Summary

| Check | Result |
|---|---|
| Revenue baseline | $41,691,716 (derived from 2026 millages × assessment roll; 2024 ACFR levy was $39,689,350, +5.1% YoY) |
| Parcel count | 34,393 city parcels (33,881 in geometry layer) |
| Census coverage | 100.0% matched |
| PNGs generated | 7 of 7 |
| SFR median change | −8.2% |
| Vacant land median change | +112.8% |

**Key results — Full LVT (land millage 52.07 mills, improvements exempt):**
- Single Family Residential: median −8.2% (48.5% of SFR parcels see >10% decrease)
- Townhome / Rowhouse: median −33.7% (land-light relative to value)
- Condominium: −100% (assessed as building-only value; TAXLND = 0)
- Large Multi-Family (5+ units): median −19.2%
- Commercial: median +15.1%
- Industrial: median +26.8%
- Vacant Land: median +112.8%

**Known limitations:**
- Revenue baseline uses 2026 adopted millages × 2024/25 assessment roll. Actual 2026 levy will differ slightly as the assessment roll updates.
- Layer 0 (assessment values) has ~34,404 parcels vs. Layer 1 geometry with ~33,881 — ~523 assessment records without matching geometry are dropped in the join.
- Current land/building split in TAXLND/TAXBLD reflects Allentown's existing assessor-assigned split.
- Full LVT models a shift from the current 5.3:1 ratio to buildings-at-zero. Under PA law (Act 511), the city may set any land-to-building ratio; building rate = 0 would require a council ordinance.